# Downloading data
Here we use data provided by [Web Data Commons](http://webdatacommons.org/largescaleproductcorpus/v2/index.html#toc4) project. They extract and structurize data from [Common Crawl](https://commoncrawl.org/).
In particular, we use `Product Data Corpus(English)` and `id-url-mapping` datasets. The first one contains data about products from different marketplaces while the second one has mappings between id and source website url

In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
df = pd.read_json('../data/translatedOffers_englishV2.json', lines=True, nrows=300000)

In [3]:
from data_processing.data_cleaning import clean_text, reformat_messy_dict, parse_list_to_dict, parse_web_url

In [4]:

df = df.drop(['specTableContent', 'cluster_id'], axis=1)

# clean columns with text data
columns_to_clean = ['category', 'title', 'description', 'keyValuePairs']
df[columns_to_clean] = df[columns_to_clean].map(clean_text)

# reformat keyValuePairs
df['keyValuePairs'] = df['keyValuePairs'].map(reformat_messy_dict)
df['identifiers'] = df['identifiers'].map(parse_list_to_dict)

# replace empty strings with NaN for easier processing
df = df.replace("", np.nan)
df = df.replace("[]", np.nan)
df = df.replace("Null", np.nan)
df = df.replace("None", np.nan)
df = df.dropna(subset=['title', 'price', 'description', 'identifiers'])
df = df.dropna(thresh=5)


Add URLs from `id_url_mapping` dataset to the corresponding products

In [5]:
target_ids = df['id'].unique()

chunks = []
for chunk in pd.read_csv('../data/id_url_mapping.csv', chunksize=100_000):
    filtered_chunk = chunk[chunk['id'].isin(target_ids)]
    chunks.append(filtered_chunk)

url_mapping = pd.concat(chunks)
df = df.merge(url_mapping, left_on='id', right_on='id')


Process URLs to names and website domains

In [6]:
df = df.drop("id", axis=1, )

df[["store name", "store url"]] = df.apply(parse_web_url, axis=1, result_type="expand")
df

,category,identifiers,title,description,brand,price,keyValuePairs,url,store name,store url
0,Tools and Home Improvement,{'/sku': 'gad00192'},"""Bee line Dragway Sign"" ""Drag Racing - Signs f...","""Bee Line Dragway in Arizona - Smell the smoke...",NaN,"""USD"", ""99.95""",NaN,http://garageart.com/products_cat2.asp?Categor...,garageart,garageart.com
1,Sports and Outdoors,{'/sku': 'underarmourftw1264304001'},"""Under Armour SpeedForm Gemini Men's Team Trai...","""Slide the UA SpeedForm Gemini on & you immedi...","""Baseball Monkey""@en","""USD""@en, ""129.99""@en",NaN,https://www.baseballmonkey.com/homerun-footwea...,baseballmonkey,www.baseballmonkey.com
2,Other Electronics,"{'/gtin8': '43233205', '/mpn': 'cese1gscu'}","""Sophos Central Endpoint Standard - competitiv...","""Sophos Cloud is the only integrated security ...",NaN,"""$"", ""22.70""",NaN,https://www.cdw.com/shop/products/Sophos-Cloud...,cdw,www.cdw.com
3,Computers and Accessories,{'/mpn': 'hardkernelodroidxu4'},"""Odroid XU4 - 8 Core Odroid computer (inc PSU)...",networking especially with faster booting USB ...,"""Hard Kernel""@en","""91.19€""@en, ""EUR""@en",NaN,https://lilliputdirect.com/index.php?_route_=o...,lilliputdirect,lilliputdirect.com
4,Sports and Outdoors,{'/sku': 'rwlbhr16h2j'},"""Rawlings Velo Two-Tone Home Junior Batting He...","""With its eye-catching finish and ultra-cushio...","""Baseball Monkey""@en","""USD""@en, ""42.99""@en",NaN,https://www.baseballmonkey.com/equipment/homer...,baseballmonkey,www.baseballmonkey.com
...,...,...,...,...,...,...,...,...,...,...
20528,Sports and Outdoors,{'/sku': 'nbbftl4040as4'},"""New Balance L4040v4 All Star Game Men's Low M...","""Who doesn't love iridescent metallic accents?...","""Baseball Monkey""@en","""USD""@en, ""149.99""@en",NaN,https://www.baseballmonkey.com/homerun-footwea...,baseballmonkey,www.baseballmonkey.com
20529,Other Electronics,"{'/gtin8': '81111812', '/mpn': 'consntas2bunk9'}","""Cisco SMARTnet extended service agreement"" "" ...","""When a problem occurs that can disrupt busine...",NaN,"""$"", ""1,216.94""",NaN,https://www.cdwg.com/shop/products/Cisco-SMART...,cdwg,www.cdwg.com
20530,Tools and Home Improvement,{'/mpn': 'oemnumberd074'},"""Handler Hollow Plastic Sprues 3/4"" per 1000""@en",""" Hollow Plastic Sprues 3/4"" Length (19.5mm) p...","""Handler""@en","""USD""@en, ""43.00""@en",NaN,https://www.dentalplanet.com/lab-equipment-mis...,dentalplanet,www.dentalplanet.com
20531,Sports and Outdoors,{'/sku': 'eastonbha168155'},"""Easton Z5 Gloss Junior Batting Helmet""@en ""Ba...",""" High impact resistant ABS shell for maximum ...","""Baseball Monkey""@en","""USD""@en, ""29.99""@en",NaN,https://www.baseballmonkey.com/equipment/homer...,baseballmonkey,www.baseballmonkey.com


In [7]:
df.to_pickle("../data/wdc_products_cleaned.pkl")

## Использовать датасет с продуктами амазаона как референс, по нему выбрать основные relations из schema(для самых частых свойств из details например). Потом использовать локальную ЛЛМку(необязательно через spacy) для того чтобы сделать entity и relation extraction. Их уже можно дополнительно включить в граф используя локальные предикаты(не из schema)